# Lab 8-1: Feature APIs

This lab demonstrates various feature APIs from MLLib, as mentioned in the lecture.


## OneHotEncoder

OneHotEncoder is a feature transformation technique that converts categorical values into binary vectors, allowing machine learning algorithms to interpret categorical data. 

In this example, we use OneHotEncoder to transform the `categoryIndex` column into a one-hot encoded vector, demonstrating how Spark handles both seen and unseen categories.

In [ ]:
from pyspark.ml.feature import OneHotEncoder
# generate the DataFrame for this example
df = spark.createDataFrame([(0.0,), (1.0,), (2.0,), (0.0,), (0.0,), (2.0,)],"categoryIndex double")
df.display()

categoryIndex
0.0
1.0
2.0
0.0
0.0
2.0

In [ ]:
ohe = OneHotEncoder(inputCol="categoryIndex", outputCol="categoryVec")
#use ohe 
df_ohe = ohe.fit(df).transform(df)
df_ohe.display()

categoryIndex | categoryVec
0.0 | {"type":"0","size":"2","indices":["0"],"values":["1.0"]}
1.0 | {"type":"0","size":"2","indices":["1"],"values":["1.0"]}
2.0 | {"type":"0","size":"2","indices":[],"values":[]}
0.0 | {"type":"0","size":"2","indices":["0"],"values":["1.0"]}
0.0 | {"type":"0","size":"2","indices":["0"],"values":["1.0"]}
2.0 | {"type":"0","size":"2","indices":[],"values":[]}

In [ ]:
df_ohe.toPandas()

Now create a OneHotEncoder instance to transform the `categoryIndex` column into a one-hot encoded vector named `categoryVec`. We then fit the encoder to the DataFrame `df` to learn the category mappings, and apply the transformation to generate the encoded vectors.

By default, OneHotEncoder drops the last category, so you would see the output column to be a size-2 vector (for 3 categories)
- Noticing that the resulting vector is a sparse vector.

Now, create a new dataframe that contains unseen categories, and use it to test how OneHotEncoder works with unseen data.

In [ ]:
df2 = spark.createDataFrame([(0.0,), (1.0,), (5.0,), (0.0,), (4.0,), (2.0,)],"categoryIndex double")

use the same encoder to encode `df2` and see what happens:

In [ ]:
df_ohe = ohe.fit(df).transform(df2)
df_ohe.display()

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File , line 2
      1 df_ohe = ohe.fit(df).transform(df2)
----> 2 df_ohe.display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:74, in apply_dataframe_display_patch..df_display(df, *args, **kwargs)
     70 def df_display(df, *args, **kwargs):
     71     """
     72     df.display() is an alias for display(df). Run help(display) for more information.
     73     """
---> 74     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
    135     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:97, in Display.

One way to handle unseen data (which you should always be prepared to deal with) is to keep them as a special category by setting `handleInvalid="keep"` when creating OneHotEncoder.

- Create a new OneHotEncoder `ohe_keep` that keeps Invalid data and test it on `df2`.

In [ ]:
ohe_keep = OneHotEncoder(inputCol="categoryIndex", outputCol="categoryVec", handleInvalid="keep")
df2_ohe_keep = ohe_keep.fit(df).transform(df2)
df2_ohe_keep.display()

categoryIndex | categoryVec
0.0 | {"type":"0","size":"3","indices":["0"],"values":["1.0"]}
1.0 | {"type":"0","size":"3","indices":["1"],"values":["1.0"]}
5.0 | {"type":"0","size":"3","indices":[],"values":[]}
0.0 | {"type":"0","size":"3","indices":["0"],"values":["1.0"]}
4.0 | {"type":"0","size":"3","indices":[],"values":[]}
2.0 | {"type":"0","size":"3","indices":["2"],"values":["1.0"]}

In [ ]:
df2_ohe_keep.toPandas()

As you can see, the new `categoryVec` has a size of 3 both the 5 and 4 are encoded as the "last" category (the vector is all-zeros because Spark drops the last category by default).

## StringIndexer & IndexToString

First, execute the following to obain a DataFrame.

In [ ]:
df = spark.createDataFrame(
    [(0, "a"), 
     (1, "b"), 
     (2, "c"), 
     (3, "a"), 
     (4, "a"), 
     (5, "c")],
    ["id", "category"])
df.display()


id | category
0 | a
1 | b
2 | c
3 | a
4 | a
5 | c

create a StringIndexer to index the `category` column and save the result in a new `cat_indexed` column in a new DataFrame `df_indexed`.

In [ ]:
from pyspark.ml.feature import StringIndexer, IndexToString
indexer = StringIndexer(inputCols=["category"], outputCols=["cat_indexed"])
df_indexed = indexer.fit(df).transform(df)
df_indexed.display()

id | category | cat_indexed
0 | a | 0.0
1 | b | 2.0
2 | c | 1.0
3 | a | 0.0
4 | a | 0.0
5 | c | 1.0

Use `IndexToString` to convert an indexed column back to string. 
- noting that `IndexToSTring`, unlike `StringIndexer`, is a transformer. It uses the meta information stored in the indexed column to back out string values.


In [ ]:
index2str = IndexToString(inputCol="cat_indexed", outputCol="cat_restored")
df_index2str = index2str.transform(df_indexed)
df_index2str.display()

id | category | cat_indexed | cat_restored
0 | a | 0.0 | a
1 | b | 2.0 | b
2 | c | 1.0 | c
3 | a | 0.0 | a
4 | a | 0.0 | a
5 | c | 1.0 | c

`OneHotEncoder` is often paired with `StringIndexer` to binarize a string category column
- `StringIndexer`: String cateogires --> numerial categories
- `OneHotEncoder`: Numerical categories --> a vector of binary indicators (dummies)

Now, define a oneHotEncoder `ohe_cat` to encode the `cat_indexed` as a new `cat_dummies` column in a new DataFrame `df_ohe`.


In [ ]:
ohe = OneHotEncoder(inputCols=["cat_indexed"], outputCols=["cat_dummies"])

# test
df_ohe = ohe.fit(df_indexed).transform(df_indexed)
df_ohe.display()


id | category | cat_indexed | cat_dummies
0 | a | 0.0 | {"type":"0","size":"2","indices":["0"],"values":["1.0"]}
1 | b | 2.0 | {"type":"0","size":"2","indices":[],"values":[]}
2 | c | 1.0 | {"type":"0","size":"2","indices":["1"],"values":["1.0"]}
3 | a | 0.0 | {"type":"0","size":"2","indices":["0"],"values":["1.0"]}
4 | a | 0.0 | {"type":"0","size":"2","indices":["0"],"values":["1.0"]}
5 | c | 1.0 | {"type":"0","size":"2","indices":["1"],"values":["1.0"]}

## SQLTransformer

`SQLTransformer(statement='your sql query')` is a powerful way of transforming your DataFrame using SQL and still make it part of the pipeline.
- here you can use Spark SQL expressions and functions, and can transform multiple columns at a time.
- you don't need to name a table to use SQLTransformer; you can use `__THIS__` to represent the underlying table of the input dataset.

Use SQLTransformer to encode the category column to add a series of binary dummies (drop the last one).

In [ ]:
from pyspark.ml.feature import SQLTransformer

st = SQLTransformer(statement="""
                    SELECT *, if(category = 'a',1,0) as cat_a, if(category = 'b',1,0) as cat_b 
                    FROM __THIS__
                    """)
    #test
df_st = st.transform(df)
df_st.display()

id | category | cat_a | cat_b
0 | a | 1 | 0
1 | b | 0 | 1
2 | c | 0 | 0
3 | a | 1 | 0
4 | a | 1 | 0
5 | c | 0 | 0

## VectorAssembler
After feature engineering, we use a VectorAssembler to convert multiple columns into a feature vector
- Use VectorAssembler to assemble `cat_a` and `cat_b` as `features`
- note that we have excluded `id` (not a feature column) and `category` (superceded by `cat_a` and `cat_b`)


In [ ]:
from pyspark.ml.feature import VectorAssembler
va = VectorAssembler(inputCols=["cat_a", "cat_b"], outputCol="features")
df_va = va.transform(df_st)
df_va.display()


id | category | cat_a | cat_b | features
0 | a | 1 | 0 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0"]}
1 | b | 0 | 1 | {"type":"1","size":null,"indices":null,"values":["0.0","1.0"]}
2 | c | 0 | 0 | {"type":"1","size":null,"indices":null,"values":["0.0","0.0"]}
3 | a | 1 | 0 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0"]}
4 | a | 1 | 0 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0"]}
5 | c | 0 | 0 | {"type":"1","size":null,"indices":null,"values":["0.0","0.0"]}

## Pipeline
The benefit of Spark's estimator/transformer approach is that you can string them together to form a repeatable pipeline.

- for example, combine the SQLTransformer and VectorAssembler steps to form a pipeline, and apply it on the DataFrame `df` that we defined earlier.


In [ ]:
from pyspark.ml import Pipeline

pipe = Pipeline(stages = [st, va])
                
#test 
df = pipe.fit(df).transform(df)
df.display()


id | category | cat_a | cat_b | features
0 | a | 1 | 0 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0"]}
1 | b | 0 | 1 | {"type":"1","size":null,"indices":null,"values":["0.0","1.0"]}
2 | c | 0 | 0 | {"type":"1","size":null,"indices":null,"values":["0.0","0.0"]}
3 | a | 1 | 0 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0"]}
4 | a | 1 | 0 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0"]}
5 | c | 0 | 0 | {"type":"1","size":null,"indices":null,"values":["0.0","0.0"]}

This is the end of the lab.